%pip install optbinning pandas xlrd pandarallel ipywidgets

In [ ]:
import pandas as pd
from pandarallel import pandarallel
from tqdm.notebook import tqdm

tqdm.pandas()


pandarallel.initialize(progress_bar=True)


INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [ ]:
data = pd.read_excel(r"/workspaces/Strategic_Segment_Builder/default of credit card clients.xls", header = 1)

In [ ]:
data.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [ ]:
data.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='str')

In [ ]:
from optbinning import OptimalBinning


In [ ]:
data['default payment next month'].value_counts(normalize = True)*100

default payment next month
0    77.88
1    22.12
Name: proportion, dtype: float64

In [ ]:
target = "default payment next month"

results = []

for col in data.columns:

    if col == target:
        continue

    try:

        dtype = (
            "categorical"
            if str(data[col].dtype) in ["object", "category"]
            else "numerical"
        )

        optb = OptimalBinning(
            name=col,
            dtype=dtype, 
            solver='cp'
        )

        optb.fit(
            data[col].values,
            data[target]
        )

        iv = list(optb.binning_table.build().IV)[-1]

        results.append({
            "variable": col,
            "iv": iv*100
        })

    except Exception as e:
        print(f"Skipping {col}: {e}")

iv_ranking = (
    pd.DataFrame(results)
      .sort_values("iv", ascending=False)
)

print(iv_ranking.head(20))

     variable         iv
6       PAY_0  86.938084
7       PAY_2  54.685883
8       PAY_3  41.242008
9       PAY_4  35.928931
10      PAY_5  33.373381
11      PAY_6  28.516449
18   PAY_AMT1  18.475602
1   LIMIT_BAL  17.983607
19   PAY_AMT2  16.504107
20   PAY_AMT3  13.207401
21   PAY_AMT4  11.124476
23   PAY_AMT6   9.852584
22   PAY_AMT5   9.285231
5         AGE   2.220189
3   EDUCATION   1.599044
16  BILL_AMT5   1.583242
17  BILL_AMT6   1.417074
15  BILL_AMT4   1.184494
0          ID   1.057210
12  BILL_AMT1   0.952468


In [ ]:
top_vars = iv_ranking.variable[0:6]

In [ ]:


binned_df = pd.DataFrame()

for col in top_vars:

    dtype = (
        "categorical"
        if str(data[col].dtype) in ["object", "category"]
        else "numerical"
    )

    optb = OptimalBinning(
        name=col,
        dtype=dtype
    )

    optb.fit(
        data[col].values,
        data[target]
    )

    binned_df[col] = optb.transform(
        data[col],
        metric="bins"
    )

binned_df[target] = data[target]

In [ ]:
binned_df.head()

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,default payment next month
0,"[1.50, inf)","[1.50, inf)","[-1.50, -0.50)","[-1.50, -0.50)","(-inf, -1.50)","(-inf, -1.50)",1
1,"(-inf, -0.50)","[1.50, inf)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[1.00, inf)",1
2,"[-0.50, 0.50)","[-0.50, 1.50)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0
3,"[-0.50, 0.50)","[-0.50, 1.50)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0
4,"(-inf, -0.50)","[-0.50, 1.50)","[-1.50, -0.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0


In [ ]:
base_rate = data['default payment next month'].mean()

In [ ]:
base_rate

np.float64(0.2212)

In [ ]:
results = []

for col in binned_df.columns:

    if col == target:
        continue

    summary = (
        binned_df
        .groupby(col)
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        col + "=" +
        summary[col].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

one_way = pd.concat(results)

In [ ]:
one_way.sort_values(["lift","rate","count"], ascending=False)

,rule,count,rate,lift
3,"PAY_0=[1.50, inf)",3130,69.552716,314.433615
3,"PAY_2=[1.50, inf)",4410,56.031746,253.308074
3,"PAY_5=[1.00, inf)",2968,55.559299,251.172239
3,"PAY_4=[0.50, inf)",3510,53.532764,242.010685
3,"PAY_6=[1.00, inf)",3079,52.322183,236.537896
3,"PAY_3=[1.50, inf)",4209,52.292706,236.404639
2,"PAY_0=[0.50, 1.50)",3688,33.947939,153.471696
0,"PAY_6=(-inf, -1.50)",4895,20.040858,90.600624
0,"PAY_5=(-inf, -1.50)",4546,19.687637,89.003786
0,"PAY_4=(-inf, -1.50)",4348,19.250230,87.026356


In [ ]:
def shortlist_rules(count,lift,rate, min_sample_size = 500, min_lift = 150):
    if count>=min_sample_size and lift>=min_lift and rate>base_rate*100:
        return 1
    else:
        return 0

    

In [ ]:
one_way['shortlist'] = one_way.parallel_apply(lambda x: shortlist_rules(x['count'],x['lift'], x['rate']), axis = 1)
one_way

0    0
1    0
2    1
3    1
0    0
1    0
2    0
3    1
0    0
1    0
2    0
3    1
0    0
1    0
2    0
3    1
0    0
1    0
2    0
3    1
0    0
1    0
2    0
3    1
dtype: int64

In [ ]:
from itertools import combinations

results = []

cols = [
    c for c in binned_df.columns
    if c != target
]

for c1, c2 in combinations(cols, 2):

    summary = (
        binned_df
        .groupby([c1, c2])
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        c1 + "=" +
        summary[c1].astype(str)
        + " & " +
        c2 + "=" +
        summary[c2].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

two_way = pd.concat(results)

In [59]:
two_way.sort_values(["lift","rate","count"], ascending=False)

,rule,count,rate,lift
1,"PAY_2=(-inf, -1.50) & PAY_3=[-0.50, 1.50)",1,1.000000,4.520796
15,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1249,0.766213,3.463892
15,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1309,0.760886,3.439811
15,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf)",1496,0.742647,3.357356
15,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf)",1754,0.728620,3.293943
...,...,...,...,...
6,"PAY_0=[-0.50, 0.50) & PAY_3=[-1.50, -0.50)",763,0.098296,0.444377
1,"PAY_4=(-inf, -1.50) & PAY_6=[-0.50, 1.00)",115,0.086957,0.393113
3,"PAY_2=(-inf, -1.50) & PAY_3=[1.50, inf)",3,0.000000,0.000000
1,"PAY_4=(-inf, -1.50) & PAY_5=[-0.50, 1.00)",3,0.000000,0.000000


In [ ]:
results = []

for c1, c2, c3 in combinations(cols, 3):

    summary = (
        binned_df
        .groupby([c1, c2, c3])
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        c1 + "=" +
        summary[c1].astype(str)
        + " & " +
        c2 + "=" +
        summary[c2].astype(str)
        + " & " +
        c3 + "=" +
        summary[c3].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

three_way = pd.concat(results)

In [61]:
three_way.sort_values(["lift","rate","count"], ascending=False)

,rule,count,rate,lift
51,"PAY_2=[1.50, inf) & PAY_4=(-inf, -1.50) & PAY_...",4,1.0,4.520796
44,"PAY_3=[1.50, inf) & PAY_4=(-inf, -1.50) & PAY_...",3,1.0,4.520796
51,"PAY_0=[1.50, inf) & PAY_3=(-inf, -1.50) & PAY_...",2,1.0,4.520796
47,"PAY_2=[1.50, inf) & PAY_3=(-inf, -1.50) & PAY_...",2,1.0,4.520796
1,"PAY_0=(-inf, -0.50) & PAY_2=(-inf, -1.50) & PA...",1,1.0,4.520796
...,...,...,...,...
9,"PAY_2=(-inf, -1.50) & PAY_4=[0.50, inf) & PAY_...",1,0.0,0.000000
6,"PAY_3=(-inf, -1.50) & PAY_4=[0.50, inf) & PAY_...",1,0.0,0.000000
7,"PAY_3=(-inf, -1.50) & PAY_4=[0.50, inf) & PAY_...",1,0.0,0.000000
12,"PAY_3=[-0.50, 1.50) & PAY_4=(-inf, -1.50) & PA...",1,0.0,0.000000
